# Task 3: Inverse Kinematics

## 3.1 Introduction
In this task, you will estimate generalized coordinates from experimental marker trajectories.

The central idea is the inverse of forward kinematics: instead of computing marker positions from known joint coordinates, you now search for the joint coordinates that best reproduce the measured marker positions.

This task builds directly on your implementation of `forward_kinematics` from assignment 1. If that solution is stored in another branch, make sure you pull it into this branch first.

## 3.2 Warm-up: Optimization with LBFGS
Before solving inverse kinematics, we first optimize a simple test problem with an autodiff library.

A standard example is the Rosenbrock function:
$$f(x, y) = (a - x)^2 + b (y - x^2)^2$$
The next cell shows how the `LBFGS` optimizer works with a closure. This is the same optimization pattern you can later use for inverse kinematics.

In [1]:
import torch

a, b = 1.0, 100.0
x = torch.tensor([-1.5, 1.5], dtype=torch.float32, requires_grad=True)
optimizer = torch.optim.LBFGS([x], max_iter=50, line_search_fn='strong_wolfe')
history = []

def rosenbrock_loss(point):
    return (a - point[0]) ** 2 + b * (point[1] - point[0] ** 2) ** 2

def closure():
    optimizer.zero_grad()
    loss = rosenbrock_loss(x)
    loss.backward()
    history.append(loss.item())
    return loss

optimizer.step(closure)
print('Optimized point:', x.detach())
print('Final loss:', history[-1])

Optimized point: tensor([1.0000, 1.0000])
Final loss: 2.48299159011367e-11


## 3.3 Set up inverse kinematics
We now use the same idea for a biomechanical model.

The experimental marker positions are stored in `data/markers.csv`. Your objective is to find generalized coordinates `q` such that the marker positions from your `forward_kinematics` function match the measured marker positions as closely as possible.

In [2]:
import numpy as np
import pandas as pd
from model.kintree import get_model_dictionary
from fk.forward_kinematics import forward_kinematics

model_dict = get_model_dictionary()
marker_data = pd.read_csv('data/markers.csv')
reference_angles = pd.read_csv('data/angles_clean.csv')
key = list(reference_angles.columns[1:])
bounds = (reference_angles[key].min().values, reference_angles[key].max().values)
single_frame = marker_data.iloc[0]

print('Number of generalized coordinates:', len(key))
print('Number of frames:', len(marker_data))

Number of generalized coordinates: 9
Number of frames: 385


**To Do:**

Implement the following functions in `ik/inverse_kinematics.py`:
- `ik_target_function`
- `compute_ik_gradient`
- `ik_solver`

Your implementation should:
- reuse your forward kinematics solution from assignment 1,
- allow `numpy.ndarray` inputs,
- skip missing markers,
- optionally add a penalty for violating joint limits.

In [3]:
from ik.inverse_kinematics import ik_target_function, compute_ik_gradient

q0 = np.zeros(len(key), dtype=float)

# After implementing the functions, this should return a finite loss and a gradient
# with the same shape as q0.
loss = ik_target_function(forward_kinematics, q0, key, model_dict, single_frame, bounds)
grad = compute_ik_gradient(forward_kinematics, q0, key, model_dict, single_frame, bounds)
print(loss)
print(np.asarray(grad).shape)

tensor(0.0432, grad_fn=<AddBackward0>)
(9,)


## 3.4 Solve inverse kinematics for the whole trial
Once the objective and gradient are implemented, solve inverse kinematics for all frames.

A common strategy is to initialize the first frame with zeros and to initialize each later frame with the solution from the previous frame.

In [4]:
from ik.inverse_kinematics import ik_solver

q_sol, mse_history = ik_solver(
     fk_function=forward_kinematics,
     kintree=model_dict,
     marker_data=marker_data,
     key=key,
     bounds=bounds,
 )
print('Number of solved frames:', len(q_sol))

  Frame    1/385  loss=0.000001
  Frame   50/385  loss=0.000007
  Frame  100/385  loss=0.000022
  Frame  150/385  loss=0.000017
  Frame  200/385  loss=0.000009
  Frame  250/385  loss=0.000010
  Frame  300/385  loss=0.000045
  Frame  350/385  loss=0.000004
Number of solved frames: 385


## 3.5 Compare your solution to the reference angles
If your implementation is correct, the recovered generalized coordinates should be close to the values in `data/angles_clean.csv`.

Plot the optimizer loss and compare a few recovered coordinates to the reference trajectories.

In [5]:
import matplotlib.pyplot as plt